# 05. Particle Filter와 Monte Carlo Localization

Particle filter는 belief를 샘플 집합으로 표현한다.

$$bel(x_t) \approx \{x_t^{[i]}, w_t^{[i]}\}_{i=1}^M$$

MCL은 motion model로 particle을 움직이고, sensor model로 weight를 계산한 뒤 resampling한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. 2D Landmark 기반 MCL

알려진 landmark까지의 거리 측정으로 로봇 위치를 추정한다.

In [ ]:
np.random.seed(8)
landmarks=np.array([[2,2],[8,2],[5,7]])
world=(10,10)
M=900
particles=np.column_stack([np.random.rand(M)*world[0], np.random.rand(M)*world[1], np.random.rand(M)*2*np.pi-np.pi])
weights=np.ones(M)/M
true=np.array([1.5,1.5,0.2])
controls=[(0.7,0.25),(0.8,0.15),(0.9,-0.2),(0.7,0.1),(0.8,0.0),(0.7,-0.25)]

def motion(p,u,noise=True):
    v,w=u
    if noise:
        v += np.random.randn()*0.08; w += np.random.randn()*0.05
    out=p.copy()
    out[0]+=v*np.cos(out[2]); out[1]+=v*np.sin(out[2]); out[2]+=w
    out[2]=np.arctan2(np.sin(out[2]),np.cos(out[2]))
    out[0]=np.clip(out[0],0,world[0]); out[1]=np.clip(out[1],0,world[1])
    return out

def systematic_resample(weights):
    M=len(weights); positions=(np.arange(M)+np.random.rand())/M
    idx=np.zeros(M,dtype=int); cumsum=np.cumsum(weights); i=j=0
    while i<M:
        if positions[i]<cumsum[j]: idx[i]=j; i+=1
        else: j+=1
    return idx

traj=[true.copy()]
frames=[]
for u in controls:
    true=motion(true,u,noise=False); traj.append(true.copy())
    for i in range(M): particles[i]=motion(particles[i],u,noise=True)
    z=np.linalg.norm(landmarks-true[:2],axis=1)+np.random.randn(len(landmarks))*0.25
    pred=np.linalg.norm(particles[:, :2][:, None, :] - landmarks[None, :, :], axis=2)
    err=pred-z[None,:]
    weights=np.exp(-0.5*np.sum((err/0.35)**2,axis=1))+1e-300
    weights/=weights.sum()
    frames.append((particles.copy(),weights.copy(),true.copy()))
    idx=systematic_resample(weights)
    particles=particles[idx]
    weights=np.ones(M)/M
traj=np.array(traj)

fig, axes=plt.subplots(2,3,figsize=(13,8),sharex=True,sharey=True)
for ax,(parts,w,true_pose),step in zip(axes.ravel(),frames,range(1,len(frames)+1)):
    ax.scatter(parts[:,0],parts[:,1],s=8,c=w,cmap='viridis',alpha=0.65)
    ax.scatter(landmarks[:,0],landmarks[:,1],marker='*',s=160,color='#1D9E75')
    ax.plot(traj[:step+1,0],traj[:step+1,1],color='black',lw=2)
    ax.scatter(true_pose[0],true_pose[1],color='#E85D24',s=80)
    ax.set_title(f'step {step}')
    ax.grid(alpha=0.2); ax.set_xlim(0,10); ax.set_ylim(0,10); ax.set_aspect('equal')
plt.tight_layout(); plt.savefig('assets/05_mcl_particles.png',dpi=150,bbox_inches='tight'); plt.show()
print('final true:', np.round(true[:2],3))
print('particle mean:', np.round(particles[:,:2].mean(axis=0),3))

## 2. Effective Sample Size

Resampling은 particle degeneracy를 막는다. 보통 다음 값이 작아지면 resampling한다.

$$N_{eff}=\frac{1}{\sum_i (w^{[i]})^2}$$

In [ ]:
ws=[np.ones(100)/100, np.r_[np.ones(10)/10, np.zeros(90)], np.array([0.7]+[0.3/99]*99)]
for w in ws:
    print('N_eff=', round(1/np.sum(w*w),2), 'of', len(w))

## 요약

| 단계 | 의미 | 책 커리큘럼 연결 |
|------|------|------------------|
| Sampling prediction | motion model 적용 | Ch.4 Nonparametric Filters, Ch.8 MCL |
| Weighting | sensor likelihood 계산 | Ch.6, Ch.8 |
| Resampling | 좋은 가설 복제, 나쁜 가설 제거 | Particle filter 핵심 |